# 51. 抖动散点图（stripplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 8 / 20 步：比较类别频数、水平与组内分布**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 小提琴图（violinplot）  →  **本章任务：** 抖动散点图（stripplot）  →  **下一步：** 蜂群图（swarmplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

当数据被分到几个类别里，光看柱状图或箱线图往往只能读到平均值和分布轮廓，很多真实的点被藏了起来。



## 本章目标

学完本章，你将能够：

- **理解**：理解「抖动散点图（stripplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「抖动散点图（stripplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「抖动散点图（stripplot）」并读出其中的结论。


## 51.1 适用场景
**背景引入**：当数据被分到几个类别里，光看柱状图或箱线图往往只能读到平均值和分布轮廓，很多真实的点被藏了起来。
抖动散点图（stripplot）会把每一个样本点单独画在分类轴上，再用微小的随机水平位移避免点完全叠死，让你既能看清每个点落在哪里，也能发现重叠和离群。
样本量不算太大、又想完整保留原始观察的时候，它比只报一个平均数要诚实得多。

打个比方：stripplot 像'把每个学生的成绩纸条摊在桌上'——它把一个个真实数据点单独摆在类别轴上，再给点左右一点'微小的随机位移'，免得纸条全叠在一起看不清。记住：那点位移只是'摆的位置'变了，纸条上的分数一个字都没改。

样本量中小，需要保留真实点并查看重叠和离群。


## 51.2 数据结构

分类变量与数值变量，每行一条观察。


## 51.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 jitter 参数从 0.22 改为 0.05 或 0.4，观察抖动幅度对点分散程度的影响
2. 调整 alpha 参数（如 0.3 或 0.9），说明透明度对重叠点可见性的作用
3. 修改 dodge=True 为 dodge=False，对比分组错位与叠加显示的视觉效果


## 51.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `orders.sample()`、`plt.subplots()`、`sns.stripplot()`、`ax.set()` | 样本量中小，需要保留真实点并查看重叠和离群。 | 点太多造成黑块 |
| 进阶变体 | `orders.sample()`、`plt.subplots()`、`sns.stripplot()`、`ax.set()` | 在基础图表上增加分组、注释、布局或交互 | 抖动过大导致类别边界模糊 |
| 关键参数 | `jitter` | 水平抖动 | 点太多造成黑块 |
| 关键参数 | `size` | 点大小 | 抖动过大导致类别边界模糊 |
| 关键参数 | `alpha` | 透明度 | 不透明点遮挡重叠 |
| 关键参数 | `dodge` | hue错位 | 点太多造成黑块 |


## 51.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(
    f"Diamonds {
        len(diamonds):,    } | Taxis {
            len(taxis):,        } | Flights {
                len(flights):,            } 行"
)


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(),
        "高于中位价",
        "不高于中位价",
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(
    f"样本：orders {
        len(orders):,    } | marketing {
            len(marketing):,        } | daily {
                len(daily):,            } 行"
)


## 51.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

sample = orders.sample(120, random_state=42)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.stripplot(
    data=sample,
    x="category",
    y="order_value",
    jitter=0.22,
    alpha=0.55,
    color="#1a73e8",
    ax=ax,
)
ax.set(title="品类客单价原始观察", xlabel="品类", ylabel="客单价（元）")
fig.tight_layout()
plt.show()


**练一练**：把 45.4 基础图表中 `sns.stripplot` 的一个参数改掉，运行后观察图形变化。

围绕“只改一个变量、看一处变化”来练手：先把 `jitter` 从 `0.22` 改成一个更大的值（如 `0.4`）或更小的值（如 `0.05`），观察点的分散程度；再把 `x="category"` 换成 `x="channel"`，看看换一个分组字段后图形怎么变。



In [ ]:
# 请在下方填写代码：复制 45.4 基础图表的 stripplot，只改 jitter（或换一个 x 字段），再运行观察。
import matplotlib.pyplot as plt
import seaborn as sns

jitter_value = 0.22  # ← 请把 0.22 改成一个新值（如 0.4 或 0.05），也可以把 x="category" 换成 x="channel"
sample = orders.sample(120, random_state=42)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.stripplot(
    data=sample,
    x="category",
    y="order_value",
    jitter=jitter_value,
    alpha=0.55,
    color="#1a73e8",
    ax=ax,
)
ax.set(title="调整 jitter 后的原始观察", xlabel="品类", ylabel="客单价（元）")
fig.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sample = orders.sample(120, random_state=42)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.stripplot(
    data=sample,
    x="category",
    y="order_value",
    jitter=0.4,  # 更大的抖动让点更分散、重叠减少；取 0.05 时点几乎挤在一条竖线上
    alpha=0.55,
    color="#1a73e8",
    ax=ax,
)
ax.set(title="增大 jitter 后的原始观察", xlabel="品类", ylabel="客单价（元）")
fig.tight_layout()
plt.show()


## 51.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

sample = orders.sample(150, random_state=420)
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.stripplot(
    data=sample,
    x="category",
    y="order_value",
    hue="channel",
    dodge=True,
    jitter=0.18,
    alpha=0.6,
    palette="colorblind",
    ax=ax,
)
ax.set(title="分渠道展示原始订单", xlabel="品类", ylabel="客单价（元）")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


## 51.8 参数说明

- jitter：水平抖动
- size：点大小
- alpha：透明度
- dodge：hue错位


## 51.9 结果解读

读取点密度、范围和异常值；抖动只改变显示位置，不改变数据。


## 51.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 51.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 51.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 51.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 51.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 51.12 易错点提醒

- 点太多造成黑块
- 抖动过大导致类别边界模糊
- 不透明点遮挡重叠


## 51.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 51.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：把 x 换成分组字段，观察换维度后的分布
# 【目标】换一个 x 分组字段，练习从不同维度看同一批数据。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：x 从 category 换成 region，看区域维度的差异。
sample = orders.sample(120, random_state=42)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.stripplot(
    data=sample,
    x="region",
    y="order_value",
    jitter=0.22,
    alpha=0.55,
    color="#188038",
    ax=ax,
)
ax.set(title="区域客单价原始观察", xlabel="区域", ylabel="客单价（元）")
fig.tight_layout()
plt.show()

# ---- 反思记录：换维度分组后，分布看法有何变化 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

sample = orders.sample(100, random_state=421)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(
    data=sample,
    x="region",
    y="order_value",
    color="white",
    showfliers=False,
    ax=ax,
)
sns.stripplot(
    data=sample,
    x="region",
    y="order_value",
    color="#188038",
    alpha=0.5,
    jitter=0.2,
    ax=ax,
)
ax.set(title="箱线摘要与原始点", xlabel="区域", ylabel="客单价（元）")
fig.tight_layout()
plt.show()


## 51.15 小结

用轻微抖动展示分类组中的每一个原始观察。


### 51.15.1 你已经掌握

- 判断抖动散点图（stripplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 51.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `jitter` | 水平抖动 |
| `size` | 点大小 |
| `alpha` | 透明度 |
| `dodge` | hue错位 |


### 51.15.3 需要注意

- 点太多造成黑块
- 抖动过大导致类别边界模糊
- 不透明点遮挡重叠


### 51.15.4 完成检查

- [ ] 能判断什么问题适合使用抖动散点图（stripplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 51.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
